# Module 2: Connecting to and Extracting Data from Multiple Sources

**Unit A · Week 2** · Track 1 — Data Integration, Standards, Metadata & Quality

Introduces this track's real, three-source practice dataset: a SQL-style extraction from a local database standing in for the hub's ERP system, and a REST/HDX-style pull, run against files instead of the live internet so the notebook is reliable to re-run anywhere. Closes with the reverse pattern — publishing a hub's own dataset as a small API.

## Learning objectives

- Query a relational database with SQL from Python and R.
- Parse a REST/JSON-shaped extract the way an HDX or HAPI pull would arrive.
- Handle extraction failure gracefully instead of letting a script fail silently.


## Setup

This notebook reads the raw practice files in `../../../data/raw/`, built by
`data/make_track1_sources.py` — three partner-style exports shaped like a
real regional hub's source-system landscape (an ERP/financial export, a
survey-platform export, and an HDX-style pull), plus the P-code gazetteer
used to reconcile them. All four are **synthetic**; see `data/README.md`.
Run `python3 data/make_track1_sources.py` once from the repo root before
working through this notebook if those files aren't there yet.

## Lesson content

See the Python notebook for the full lesson content — identical in both languages.

In [ ]:
library(DBI)
library(RSQLite)
library(readr)
library(dplyr)

# --- SQL extraction, simulating the hub's ERP/financial system ---
partner_a <- read_csv("../../../data/raw/partner_a_finance_export.csv", show_col_types = FALSE)
con <- dbConnect(SQLite(), ":memory:")
dbWriteTable(con, "finance_export", partner_a)

db_df <- dbGetQuery(con, "SELECT reg, rev, date FROM finance_export WHERE rev >= ?", params = list(30.0))
dbDisconnect(con)
cat(nrow(db_df), "rows extracted via SQL (rev >= 30.0)\n")
head(db_df)

In [ ]:
# --- "API" extraction, simulating an HDX/HAPI-style pull ---
fetch_hdx_style <- function(path, timeout_ok = TRUE) {
  if (!timeout_ok) stop("simulated HDX endpoint timeout")
  read_csv(path, show_col_types = FALSE)
}

api_df <- tryCatch(
  fetch_hdx_style("../../../data/raw/partner_c_hdx_pull.csv"),
  error = function(e) {
    message("Extraction failed, logged for retry: ", conditionMessage(e))  # never fail silently
    tibble()
  }
)
head(api_df)

### Module 2 Extension — worked example: serving integrated data as an API

The R equivalent of FastAPI is **plumber**: ordinary functions turned into HTTP endpoints via lightweight `#*` annotations. Illustrative only — run from a terminal, not inline here.

In [ ]:
# api.R — run with: plumber::pr("api.R") |> plumber::pr_run(port = 8000)  (not executed here)
library(plumber)
library(dplyr)
library(readr)

load_latest <- function() {
  read_csv("../../../data/processed/track2_dataset.csv", show_col_types = FALSE)
}

#* List all district P-codes and names currently published
#* @get /districts
function() {
  load_latest() %>% distinct(district_pcode, district)
}

#* Return published records for a single district, looked up by P-code
#* @param pcode The district P-code
#* @get /districts/<pcode>
function(pcode, res) {
  result <- load_latest() %>% filter(district_pcode == pcode)
  if (nrow(result) == 0) {
    res$status <- 404
    return(list(detail = paste0("P-code '", pcode, "' not found")))
  }
  result
}

# Interactive docs are generated automatically at /__docs__/

## Your turn

Extract the finance export via a SQL query and the HDX-style pull via the file-based simulation above, in both Python and R, and confirm the failure-handling branch logs rather than crashes when `timeout_ok=False`.

**Formative assessment.** Submitted script plus resulting extracted tables, graded on correct filtering, and on graceful (not silent) handling of the simulated timeout.

## Governance / responsibility callback

The discussion prompt for the API extension: what access control would this API need before it could be exposed outside the hub's internal network — and which Module 10 governance practices (licensing, sensitive-field review, an accountable owner) apply just as much to an API endpoint as to a shared CSV file?